In [6]:
import os
import json
from pathlib import Path
from datetime import date

# Find prompts file
prompts_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if "week2_prompts" in f and f.endswith(".txt"):
            prompts_path = Path(root) / f
            break

print(f"Prompts: {prompts_path}")

with open(prompts_path) as f:
    prompt_lines = [line.strip() for line in f 
                    if line.strip() and not line.strip().startswith("#") and "|" in line]

prompts_lookup = {}
for line in prompt_lines:
    pid, category, prompt_text = line.split("|", 2)
    prompts_lookup[pid] = {
        "category": category,
        "prompt_text": prompt_text,
    }

CINEMATIC_SUFFIX = ", natural lighting, cinematic quality, high detail, realistic"
print(f"Loaded {len(prompts_lookup)} prompts")

# Find video directories
print("\nSearching for video files...")
for root, dirs, files in os.walk("/kaggle/input"):
    mp4s = [f for f in files if f.endswith('.mp4')]
    if mp4s:
        print(f"  {root} ({len(mp4s)} videos)")

Prompts: /kaggle/input/datasets/shantanuvedanteog/week2-prompts-v1generation-txt/week2_prompts_v1(generation).txt
Loaded 40 prompts

Searching for video files...
  /kaggle/input/datasets/shantanuvedanteog/commercial-renamed/commercial/gemini_omni_flash (32 videos)
  /kaggle/input/datasets/shantanuvedanteog/commercial-renamed/commercial/kling (32 videos)
  /kaggle/input/datasets/shantanuvedanteog/commercial-renamed/commercial/seedance (32 videos)
  /kaggle/input/datasets/shantanuvedanteog/pexels-real/real_videos_pexels (64 videos)


In [8]:
COMMERCIAL_ROOT = Path("/kaggle/input")  # will be refined after Cell 1

# Find each generator's folder automatically
generator_folders = {}
for root, dirs, files in os.walk("/kaggle/input"):
    mp4s = [f for f in files if f.endswith('.mp4')]
    if mp4s:
        folder_name = Path(root).name
        if any(f.startswith("w2_") and "_kling" in f for f in mp4s):
            generator_folders["kling"] = Path(root)
        elif any(f.startswith("w2_") and "_seedance" in f for f in mp4s):
            generator_folders["seedance"] = Path(root)
        elif any(f.startswith("w2_") and "_gemini" in f for f in mp4s):
            generator_folders["gemini_omni_flash"] = Path(root)
        elif any(f.startswith("pexels_") for f in mp4s):
            generator_folders["pexels"] = Path(root)

print("Found folders:")
for name, path in generator_folders.items():
    count = len(list(path.glob("*.mp4")))
    print(f"  {name}: {path} ({count} videos)")

OUTPUT_ROOT = Path("/kaggle/working/metadata")
OUTPUT_ROOT.mkdir(exist_ok=True)

Found folders:
  gemini_omni_flash: /kaggle/input/datasets/shantanuvedanteog/commercial-renamed/commercial/gemini_omni_flash (32 videos)
  kling: /kaggle/input/datasets/shantanuvedanteog/commercial-renamed/commercial/kling (32 videos)
  seedance: /kaggle/input/datasets/shantanuvedanteog/commercial-renamed/commercial/seedance (32 videos)
  pexels: /kaggle/input/datasets/shantanuvedanteog/pexels-real/real_videos_pexels (64 videos)


In [11]:
# Metadata template per source
source_configs = {
    "kling": {
        "generator_name": "Kling 3.0",
        "access_method": "Higgsfield trial",
        "native_resolution": "1280x720",
        "native_duration_sec": 3,
        "native_fps": 24,
        "native_aspect": "16:9",
        "has_audio_native": False,
        "generation_date": "2026-07-21",
        "license": "Generated via Higgsfield trial tier; Kling 3.0 by Kuaishou",
        "source_type": "commercial",
        "filename_pattern": "w2_XXX_kling.mp4",
    },
    "gemini_omni_flash": {
        "generator_name": "Gemini Omni Flash",
        "access_method": "Higgsfield trial",
        "native_resolution": "1280x720",
        "native_duration_sec": 3,
        "native_fps": 24,
        "native_aspect": "16:9",
        "has_audio_native": True,
        "generation_date": "2026-07-21",
        "license": "Generated via Higgsfield trial tier; Gemini Omni Flash by Google",
        "source_type": "commercial",
        "filename_pattern": "w2_XXX_gemini_omni_flash.mp4",
    },
    "seedance": {
        "generator_name": "Seedance 2.0",
        "access_method": "Higgsfield trial",
        "native_resolution": "1280x720",
        "native_duration_sec": 4,
        "native_fps": 24,
        "native_aspect": "16:9",
        "has_audio_native": False,
        "generation_date": "2026-07-22",
        "license": "Generated via Higgsfield trial tier; Seedance 2.0 by ByteDance",
        "source_type": "commercial",
        "filename_pattern": "w2_XXX_seedance.mp4",
    },
    "pexels": {
        "generator_name": "Real video (not AI-generated)",
        "access_method": "Pexels API (free tier)",
        "native_resolution": "1280x720 or higher",
        "native_duration_sec": "3-30 (trimmed to 3)",
        "native_fps": "variable (normalised to 24)",
        "native_aspect": "mostly 16:9",
        "has_audio_native": "variable (stripped during normalisation)",
        "generation_date": "N/A (real video)",
        "collection_date": "2026-07-20",
        "license": "Pexels License (free for commercial and research use, no attribution required)",
        "source_type": "real_baseline",
        "filename_pattern": "pexels_CATEGORY_ID.mp4",
    },
}

In [12]:
import subprocess

def get_video_info(video_path):
    """Get actual video specs via ffprobe."""
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "stream=width,height,r_frame_rate,duration,codec_name",
        "-show_entries", "format=size,duration,bit_rate",
        "-of", "json",
        str(video_path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        try:
            data = json.loads(result.stdout)
            stream = data.get("streams", [{}])[0]
            fmt = data.get("format", {})
            fps_parts = stream.get("r_frame_rate", "0/1").split("/")
            fps = round(int(fps_parts[0]) / int(fps_parts[1]), 2) if len(fps_parts) == 2 and int(fps_parts[1]) > 0 else 0
            return {
                "actual_width": stream.get("width"),
                "actual_height": stream.get("height"),
                "actual_fps": fps,
                "actual_duration_sec": round(float(stream.get("duration", fmt.get("duration", 0))), 2),
                "actual_codec": stream.get("codec_name"),
                "actual_filesize_bytes": int(fmt.get("size", 0)),
                "actual_bitrate_bps": int(fmt.get("bit_rate", 0)),
            }
        except:
            pass
    return {}


for gen_key in ["kling", "gemini_omni_flash", "seedance"]:
    if gen_key not in generator_folders:
        print(f"SKIP {gen_key}: folder not found")
        continue
    
    config = source_configs[gen_key]
    gen_folder = generator_folders[gen_key]
    out_folder = OUTPUT_ROOT / gen_key
    out_folder.mkdir(exist_ok=True)
    
    videos = sorted(gen_folder.glob("*.mp4"))
    print(f"\n=== {config['generator_name']} ({len(videos)} videos) ===")
    
    all_metadata = []
    
    for v in videos:
        # Extract prompt_id from filename: w2_001_kling.mp4 -> w2_001
        parts = v.stem.rsplit("_", 1)  # split from right: w2_001_kling -> ['w2_001', 'kling'] or similar
        # More robust: find the w2_XXX pattern
        import re
        pid_match = re.match(r'(w2_\d+)', v.stem)
        if not pid_match:
            print(f"  SKIP {v.name}: no prompt_id found")
            continue
        
        prompt_id = pid_match.group(1)
        prompt_info = prompts_lookup.get(prompt_id)
        
        if not prompt_info:
            print(f"  SKIP {v.name}: prompt_id {prompt_id} not in prompts file")
            continue
        
        # Get actual video specs
        video_info = get_video_info(v)
        
        metadata = {
            "video_id": v.name,
            "prompt_id": prompt_id,
            "category": prompt_info["category"],
            "original_prompt": prompt_info["prompt_text"],
            "full_prompt_with_suffix": prompt_info["prompt_text"] + CINEMATIC_SUFFIX,
            "generator": config["generator_name"],
            "source_type": config["source_type"],
            "access_method": config["access_method"],
            "generation_date": config["generation_date"],
            "license": config["license"],
            "native_specs": {
                "resolution": config["native_resolution"],
                "duration_sec": config["native_duration_sec"],
                "fps": config["native_fps"],
                "aspect_ratio": config["native_aspect"],
                "has_audio": config["has_audio_native"],
            },
            "actual_file_specs": video_info,
        }
        
        all_metadata.append(metadata)
        
        # Save per-video JSON
        json_path = out_folder / f"{v.stem}.json"
        with open(json_path, "w") as f:
            json.dump(metadata, f, indent=2)
    
    # Save combined JSON for this generator
    combined_path = OUTPUT_ROOT / f"{gen_key}_metadata.json"
    with open(combined_path, "w") as f:
        json.dump(all_metadata, f, indent=2)
    
    print(f"  Saved {len(all_metadata)} individual JSONs to {out_folder}/")
    print(f"  Saved combined to {combined_path}")


=== Kling 3.0 (32 videos) ===
  Saved 32 individual JSONs to /kaggle/working/metadata/kling/
  Saved combined to /kaggle/working/metadata/kling_metadata.json

=== Gemini Omni Flash (32 videos) ===
  Saved 32 individual JSONs to /kaggle/working/metadata/gemini_omni_flash/
  Saved combined to /kaggle/working/metadata/gemini_omni_flash_metadata.json

=== Seedance 2.0 (32 videos) ===
  Saved 32 individual JSONs to /kaggle/working/metadata/seedance/
  Saved combined to /kaggle/working/metadata/seedance_metadata.json


In [13]:
if "pexels" not in generator_folders:
    print("Pexels folder not found — check Cell 1 output")
else:
    config = source_configs["pexels"]
    pexels_folder = generator_folders["pexels"]
    out_folder = OUTPUT_ROOT / "pexels"
    out_folder.mkdir(exist_ok=True)
    
    videos = sorted(pexels_folder.glob("*.mp4"))
    print(f"=== Pexels real videos ({len(videos)} videos) ===")
    
    # Pexels category is embedded in filename: pexels_portrait_12345.mp4
    all_metadata = []
    
    for v in videos:
        # Extract category and pexels_id from filename
        # Pattern: pexels_CATEGORY_ID.mp4
        # e.g. pexels_portrait_5045936.mp4 -> category=portrait, pexels_id=5045936
        name_parts = v.stem.split("_")
        # First part is always "pexels", last part is the ID, middle parts are the category
        if len(name_parts) < 3:
            print(f"  SKIP {v.name}: unexpected filename format")
            continue
        
        pexels_id = name_parts[-1]
        category = "_".join(name_parts[1:-1])  # handles multi_person, text_scene etc.
        
        video_info = get_video_info(v)
        
        metadata = {
            "video_id": v.name,
            "pexels_id": pexels_id,
            "category": category,
            "generator": config["generator_name"],
            "source_type": config["source_type"],
            "access_method": config["access_method"],
            "collection_date": config["collection_date"],
            "pexels_page_url": f"https://www.pexels.com/video/{pexels_id}/",
            "license": config["license"],
            "normalisation_applied": {
                "target_resolution": "832x480",
                "target_fps": 24,
                "target_duration_sec": 3,
                "codec": "libx264",
                "crf": 18,
                "preset": "slow",
                "audio": "stripped",
                "aspect_handling": "letterbox with black padding",
            },
            "actual_file_specs": video_info,
        }
        
        all_metadata.append(metadata)
        
        # Per-video JSON
        json_path = out_folder / f"{v.stem}.json"
        with open(json_path, "w") as f:
            json.dump(metadata, f, indent=2)
    
    combined_path = OUTPUT_ROOT / "pexels_metadata.json"
    with open(combined_path, "w") as f:
        json.dump(all_metadata, f, indent=2)
    
    print(f"  Saved {len(all_metadata)} individual JSONs to {out_folder}/")
    print(f"  Saved combined to {combined_path}")

=== Pexels real videos (64 videos) ===
  Saved 64 individual JSONs to /kaggle/working/metadata/pexels/
  Saved combined to /kaggle/working/metadata/pexels_metadata.json


In [14]:
from collections import Counter

print("=" * 60)
print("METADATA SUMMARY")
print("=" * 60)

total = 0
for source in ["kling", "gemini_omni_flash", "seedance", "pexels"]:
    combined_path = OUTPUT_ROOT / f"{source}_metadata.json"
    if combined_path.exists():
        with open(combined_path) as f:
            data = json.load(f)
        
        categories = Counter(item["category"] for item in data)
        print(f"\n{source}: {len(data)} videos")
        for cat, count in sorted(categories.items()):
            print(f"  {cat}: {count}")
        total += len(data)
    else:
        print(f"\n{source}: NO METADATA FILE FOUND")

print(f"\n{'='*60}")
print(f"TOTAL: {total} videos with metadata")

# Verify per-video JSONs exist
per_video_count = 0
for source in ["kling", "gemini_omni_flash", "seedance", "pexels"]:
    source_dir = OUTPUT_ROOT / source
    if source_dir.exists():
        jsons = list(source_dir.glob("*.json"))
        per_video_count += len(jsons)

print(f"Per-video JSONs: {per_video_count}")
print(f"\nAll files in {OUTPUT_ROOT}:")
for item in sorted(OUTPUT_ROOT.rglob("*")):
    if item.is_file():
        print(f"  {item.relative_to(OUTPUT_ROOT)}")

METADATA SUMMARY

kling: 32 videos
  animal: 4
  edge_case: 4
  hands: 4
  motion: 4
  multi_person: 4
  portrait: 4
  text_scene: 4
  texture: 4

gemini_omni_flash: 32 videos
  animal: 4
  edge_case: 4
  hands: 4
  motion: 4
  multi_person: 4
  portrait: 4
  text_scene: 4
  texture: 4

seedance: 32 videos
  animal: 4
  edge_case: 4
  hands: 4
  motion: 4
  multi_person: 4
  portrait: 4
  text_scene: 4
  texture: 4

pexels: 64 videos
  animal: 8
  edge_case: 8
  hands: 8
  motion: 8
  multi_person: 8
  portrait: 8
  text_scene: 8
  texture: 8

TOTAL: 160 videos with metadata
Per-video JSONs: 160

All files in /kaggle/working/metadata:
  gemini_omni_flash/w2_001_gemini_omni_flash.json
  gemini_omni_flash/w2_002_gemini_omni_flash.json
  gemini_omni_flash/w2_003_gemini_omni_flash.json
  gemini_omni_flash/w2_004_gemini_omni_flash.json
  gemini_omni_flash/w2_006_gemini_omni_flash.json
  gemini_omni_flash/w2_007_gemini_omni_flash.json
  gemini_omni_flash/w2_008_gemini_omni_flash.json
  gemin

In [15]:
import shutil

shutil.make_archive("/kaggle/working/all_metadata", "zip", str(OUTPUT_ROOT))
!ls -la /kaggle/working/*.zip
print("\nDownload all_metadata.zip from the Output panel.")
print("Extract to corpus/metadata/ in your local repo.")

-rw-r--r-- 1 root root 108012 Jul 23 13:44 /kaggle/working/all_metadata.zip

Download all_metadata.zip from the Output panel.
Extract to corpus/metadata/ in your local repo.
